# Support Ticket Triage — Multi-Agent Workflow
## AI Agents and Workflows for Developers · April 2026 · Individual Project

### Scenario
A customer support team receives raw support tickets. This notebook implements a
**stateful multi-agent LangGraph workflow** that automatically:

1. Pre-loads per-user long-term memory (preferences, escalation history)
2. **Triages** the ticket into structured metadata (category, priority, sentiment, entities)
3. **Retrieves** relevant KB evidence, account context, and urgency signals via tools
4. **Plans** a route and recommended action informed by KB + memory + account context
5. **Drafts** a grounded, tone-aware customer reply
6. **Pauses for human review** (HITL) — approve / revise / manual escalation
7. **Finalises** and writes updated facts back to long-term memory

### Architecture
- **LangGraph** stateful `StateGraph` with SQLite persistence
- **5 agent nodes**: Memory Loader · Triage · Retrieval · Resolution · Drafting
- **1 HITL node**: Review (interrupt/resume)
- **5 tools**: `search_kb` · `lookup_account_context` · `detect_escalation_risk` · `priority_score` · `route_policy_lookup`
- **3-layer memory**: short-term (messages + tiktoken trim) · long-term (SQLite namespaced) · RAG (FAISS over 17 KB articles)
- **7 test cases** covering all HITL paths: approve · revise · escalate_manually · clarify


## Cell 1 — Package Installation

In [ ]:
%pip install -q --upgrade-strategy only-if-needed \
  "langchain==0.3.16" \
  "langgraph==0.2.76" \
  "langgraph-checkpoint-sqlite==2.0.6" \
  "langchain-openai==0.2.11" \
  "langchain-community==0.3.16" \
  "faiss-cpu==1.8.0" \
  "pydantic==2.12.3" \
  "tiktoken==0.7.0" \
  "numpy==1.26.4" \
  "requests==2.32.4"

import os
import sys

def _maybe_restart_colab_once(marker_path: str = "/content/.triage_deps_restart_done") -> None:
    """Restart Colab once after installing native wheels to avoid first-run FAISS failures."""
    if "google.colab" not in sys.modules:
        print("Dependency install complete (non-Colab runtime).")
        return

    if os.path.exists(marker_path):
        print("Dependency install complete (Colab restart already done).")
        return

    with open(marker_path, "w", encoding="utf-8") as f:
        f.write("ok")

    print("First-time Colab dependency install detected.")
    print("Restarting runtime to stabilize FAISS/native wheels — re-run notebook from Cell 1 after restart.")

    # get_ipython() is always available in Colab's kernel and holds a live .kernel reference.
    # IPython.Application.instance() is NOT reliable in Colab (Application may be uninitialised).
    shell = get_ipython()
    if shell is not None and hasattr(shell, "kernel"):
        shell.kernel.do_shutdown(restart=True)
    else:
        # Fallback: print a manual instruction and do nothing destructive.
        os.remove(marker_path)   # reset marker so the prompt appears next time
        print("⚠️  Automatic restart unavailable. Please restart the runtime manually:")
        print("    Runtime → Restart session   (then re-run from Cell 1)")

_maybe_restart_colab_once()


In [ ]:
# Optional sanity check cell (kept lightweight).
import importlib

for module_name in [
    "langchain",
    "langgraph",
    "langchain_openai",
    "langchain_community",
    "faiss",
]:
    importlib.import_module(module_name)

print("Dependency import sanity check passed.")

## Cell 2 — API Key Setup

In [ ]:
import os

try:
    from google.colab import userdata
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # Running outside Colab — key must already be in environment

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is required. "
        "Add it via Colab Secrets (key icon) or set it as an environment variable."
    )

print("API key loaded successfully.")


## Cell 3 — Imports

In [ ]:
import os
import re
import json
import uuid
import time
import sqlite3
from typing import TypedDict, List, Dict, Optional, Literal, Any

import tiktoken
from pydantic import BaseModel, Field

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.sqlite import SqliteSaver

print("All imports successful.")


## Cell 4 — Load Project Data and Constants

In [ ]:
DATA_FILE = "https://raw.githubusercontent.com/vjfrosty/Sofuni_agents_support_ticket_triage_submission/main/project_data.json"


def load_project_data(url: str = DATA_FILE) -> dict:
    """Load project data from GitHub repository."""
    import urllib.request
    try:
        with urllib.request.urlopen(url) as response:
            data = json.loads(response.read().decode('utf-8'))
            print(f"✓ Data loaded successfully from GitHub: {url}")
            return data
    except Exception as e:
        raise RuntimeError(f"Failed to load project_data.json from GitHub: {e}")


PROJECT_DATA = load_project_data()

# Wire settings from JSON so they are configurable without touching code
_s = PROJECT_DATA.get("settings", {})
EMBEDDING_MODEL     = _s.get("embedding_model",       "text-embedding-3-small")
RETRIEVER_TOP_K     = _s.get("retriever_top_k",       4)
MSG_TRIM_MAX_TOKENS = _s.get("message_trim_max_tokens", 1800)
DEFAULT_TONE        = _s.get("default_tone",           "professional_empathetic")

print("Settings loaded:")
print(f"  embedding_model      = {EMBEDDING_MODEL}")
print(f"  retriever_top_k      = {RETRIEVER_TOP_K}")
print(f"  msg_trim_max_tokens  = {MSG_TRIM_MAX_TOKENS}")
print(f"  default_tone         = {DEFAULT_TONE}")
print()
print(f"KB articles     : {len(PROJECT_DATA['kb_articles'])}")
print(f"Accounts        : {len(PROJECT_DATA['account_context'])}")
print(f"Routing rules   : {len(PROJECT_DATA['routing_rules'])}")
print(f"Test cases      : {len(PROJECT_DATA['test_cases'])}")


## Cell 5 — RAG Pipeline: KB → Documents → FAISS Vector Store

In [ ]:
def kb_to_documents(kb_articles: List[dict]) -> List[Document]:
    return [
        Document(
            page_content=article["text"],
            metadata={
                "id":       article["id"],
                "title":    article["title"],
                "category": article.get("category", ""),
                "tags":     article.get("tags", []),
            },
        )
        for article in kb_articles
    ]


def build_chunks(documents: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    return splitter.split_documents(documents)


def build_retriever(kb_articles: List[dict]):
    docs       = kb_to_documents(kb_articles)
    chunks     = build_chunks(docs)
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever  = vectorstore.as_retriever(search_kwargs={"k": RETRIEVER_TOP_K})
    return retriever, vectorstore, chunks


print("Building FAISS vector store from KB articles...")
try:
    RETRIEVER, VECTORSTORE, KB_CHUNKS = build_retriever(PROJECT_DATA["kb_articles"])
except Exception as e:
    raise RuntimeError(
        "FAISS initialization failed. If running in Colab right after first install, "
        "run Cell 1 to let it perform one-time restart, then re-run notebook from top. "
        f"Original error: {e}"
    ) from e

print(f"Vector store ready: {len(KB_CHUNKS)} chunks from {len(PROJECT_DATA['kb_articles'])} articles.")


## Cell 6 — Account Context, Routing Rules, Test Cases

In [ ]:
ACCOUNT_CONTEXT = PROJECT_DATA.get("account_context", {})
ROUTING_RULES   = PROJECT_DATA.get("routing_rules",   [])
TEST_CASES      = PROJECT_DATA.get("test_cases",       [])
MEMORY_DEFAULTS = PROJECT_DATA.get("memory_defaults",  {})

print(f"Account contexts loaded : {len(ACCOUNT_CONTEXT)}")
print(f"Routing rules loaded    : {len(ROUTING_RULES)}")
print(f"Test cases loaded       : {len(TEST_CASES)}")
print(f"Memory defaults loaded  : {len(MEMORY_DEFAULTS)} namespaces")


## Cell 7 — Long-Term Memory: SQLite `memory_store`

Three distinct memory layers:

| Layer | Mechanism | Scope |
|---|---|---|
| **Short-term** | `messages` field in `TicketState` + tiktoken trim | Per thread |
| **Long-term** | SQLite `memory_store` table (this cell) | Per user / global |
| **RAG** | FAISS vector store (Cell 5) | Static KB |


In [ ]:
MEMORY_DB_PATH = "memory_store.sqlite"


def init_memory_db(path: str = MEMORY_DB_PATH) -> sqlite3.Connection:
    conn = sqlite3.connect(path, check_same_thread=False)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS memory_store (
            namespace  TEXT NOT NULL,
            key        TEXT NOT NULL,
            value      JSON NOT NULL,
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (namespace, key)
        )
    """)
    conn.commit()
    return conn


def load_memory(conn: sqlite3.Connection, namespace: str) -> dict:
    """Read all key-value pairs for a namespace from long-term memory."""
    try:
        cursor = conn.execute(
            "SELECT key, value FROM memory_store WHERE namespace = ?", (namespace,)
        )
        return {row[0]: json.loads(row[1]) for row in cursor.fetchall()}
    except Exception as e:
        print(f"[WARNING] load_memory('{namespace}'): {e}")
        return {}


def save_memory(conn: sqlite3.Connection, namespace: str, key: str, value: Any):
    """Upsert a single key-value pair into long-term memory."""
    try:
        conn.execute(
            """INSERT OR REPLACE INTO memory_store (namespace, key, value, updated_at)
               VALUES (?, ?, ?, CURRENT_TIMESTAMP)""",
            (namespace, key, json.dumps(value)),
        )
        conn.commit()
    except Exception as e:
        print(f"[WARNING] save_memory('{namespace}', '{key}'): {e}")


def seed_memory_defaults(conn: sqlite3.Connection, memory_defaults: dict):
    """
    Seed the memory store with pre-defined defaults from project_data.json.
    Each top-level key is a namespace (e.g. 'user:acct_1001:preferences').
    Each nested key-value pair becomes a row.
    Uses INSERT OR IGNORE so existing live data is never overwritten.
    """
    count = 0
    for namespace, fields in memory_defaults.items():
        if isinstance(fields, dict):
            for key, value in fields.items():
                try:
                    conn.execute(
                        """INSERT OR IGNORE INTO memory_store (namespace, key, value, updated_at)
                           VALUES (?, ?, ?, CURRENT_TIMESTAMP)""",
                        (namespace, key, json.dumps(value)),
                    )
                    count += 1
                except Exception as e:
                    print(f"[WARNING] seed_memory failed for {namespace}/{key}: {e}")
    conn.commit()
    print(f"Memory defaults seeded: {count} entries across {len(memory_defaults)} namespaces.")


MEMORY_CONN = init_memory_db()
seed_memory_defaults(MEMORY_CONN, MEMORY_DEFAULTS)
print("Long-term memory DB ready.")


## Cell 8 — Short-Term Memory: `prepare_messages`

Uses **tiktoken** to count tokens and trim the `messages` list before every LLM call.
Works with plain `{role, content}` dicts — no LangChain message object conversion required.


In [ ]:
_TIKTOKEN_ENC = tiktoken.get_encoding("cl100k_base")


def _count_tokens(message: dict) -> int:
    return len(_TIKTOKEN_ENC.encode(str(message.get("content", ""))))


def prepare_messages(messages: List[dict]) -> List[dict]:
    """
    Trim the messages list so the total token count stays within MSG_TRIM_MAX_TOKENS.
    Keeps the most recent messages. Falls back to the last 10 messages on error.
    """
    if not messages:
        return []
    try:
        total, trimmed = 0, []
        for m in reversed(messages):
            tokens = _count_tokens(m)
            if total + tokens > MSG_TRIM_MAX_TOKENS and trimmed:
                break
            trimmed.insert(0, m)
            total += tokens
        return trimmed if trimmed else messages[-1:]
    except Exception as e:
        print(f"[WARNING] prepare_messages failed: {e}. Using last 10 messages.")
        return messages[-10:]


print(f"prepare_messages ready (max_tokens={MSG_TRIM_MAX_TOKENS}).")


## Cell 9 — State Definition and Structured Output Schemas

In [ ]:
class TicketState(TypedDict, total=False):
    # --- identity ---
    thread_id: str
    user_request: str

    # --- SHORT-TERM MEMORY (append-only; trimmed per LLM call) ---
    messages: List[dict]          # [{"role": "user"|"assistant", "content": "..."}]

    # --- PRE-LOADED LONG-TERM MEMORY (populated by load_memory_node) ---
    memory_context: Dict[str, Any]

    # --- triage output ---
    category: str
    subcategory: str
    priority: str
    sentiment: str
    requires_escalation: bool
    missing_info: List[str]
    extracted_entities: Dict[str, str]   # always includes "account_id" if detectable

    # --- retrieval output ---
    kb_results: List[Dict[str, Any]]
    kb_summary: str
    account_context: Dict[str, Any]

    # --- resolution output ---
    route_to_team: str
    recommended_action: str
    internal_notes: str
    should_ask_for_clarification: bool

    # --- draft / final ---
    reply_draft: str
    final_reply: str

    # --- HITL ---
    review_action: Optional[str]
    human_feedback: Optional[str]
    human_approved: bool

    # --- meta ---
    status: str
    errors: List[str]


# NOTE: long-term memory (memory_store) is NOT stored in TicketState.
# It lives in SQLite and is accessed via MEMORY_CONN at node boundaries.


class TriageOutput(BaseModel):
    category: str = Field(
        description=(
            "Primary support category. One of: "
            "billing, technical, account_access, cancellation, complaint, service_request, unknown"
        )
    )
    subcategory: str = Field(description="Specific issue type within the category.")
    priority: str = Field(description="Urgency level: low, medium, high, or critical.")
    sentiment: str = Field(description="Customer tone: neutral, frustrated, or angry.")
    requires_escalation: bool = Field(
        description="True if the ticket shows strong escalation signals."
    )
    missing_info: List[str] = Field(
        default_factory=list,
        description="List of specific information items missing from the ticket.",
    )
    extracted_entities: Dict[str, str] = Field(
        default_factory=dict,
        description=(
            "Key entities extracted from the ticket text. "
            "MUST include 'account_id' mapped to the account identifier "
            "(e.g. 'acct_1001') if any such pattern appears in the text."
        ),
    )


class ResolutionPlan(BaseModel):
    route_to_team: str = Field(description="Team to route this ticket to.")
    recommended_action: str = Field(description="Specific action for the receiving team.")
    internal_notes: str = Field(description="Concise internal notes for the team.")
    should_ask_for_clarification: bool = Field(
        description="True if more information is needed before acting."
    )


print("TicketState, TriageOutput, ResolutionPlan defined.")


## Cell 10 — LLM Setup

In [ ]:
base_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def make_structured_llm(schema):
    """
    Create a structured-output LLM.
    Prefers function_calling: no strict-mode restrictions, compatible with
    Dict fields and optional/default_factory fields.
    Falls back to json_schema for environments where function_calling is unavailable.
    """
    for method in ("function_calling", "json_schema"):
        try:
            return base_llm.with_structured_output(schema, method=method)
        except Exception as e:
            print(f"[INFO] with_structured_output method='{method}' failed: {e}")
    raise RuntimeError(f"Could not create structured LLM for schema '{schema.__name__}'.")


triage_llm = make_structured_llm(TriageOutput)
plan_llm   = make_structured_llm(ResolutionPlan)

print("LLMs ready: base_llm (gpt-4o-mini), triage_llm, plan_llm.")


## Cell 11 — Prompt Library

In [ ]:
TRIAGE_SYSTEM = """
You are a support triage specialist. Convert a raw customer support ticket into structured metadata.

Output fields:
- category   : billing | technical | account_access | cancellation | complaint | service_request | unknown
- subcategory: specific issue type
- priority   : low | medium | high | critical
- sentiment  : neutral | frustrated | angry
- requires_escalation: true/false
- missing_info: list of specific items needed to safely classify or resolve the ticket
- extracted_entities: key entities — ALWAYS map "account_id" to the account identifier
  (e.g. "acct_1001") if any pattern matching "acct_XXXX" appears anywhere in the text.

Classification rules (apply strictly):
- Cancellation language with anger or urgency → category=cancellation, priority=high, requires_escalation=true
- Legal / regulatory mentions (regulator, ombudsman, lawyer, authority, legal action) →
    requires_escalation=true, priority=critical, category=complaint
- Business-tier outage affecting multiple users → priority=critical
- requires_escalation=true ONLY when one of these is explicitly present:
    * Customer states a PREVIOUS support ticket or request was ignored / not resolved (e.g. "nobody replied", "still not fixed", "my ticket is still open")
    * Duplicate or incorrect charge (e.g. "charged twice", "wrong amount")
    * NOTE: "still cannot do X" or "still experiencing Y" after a self-service step does NOT trigger escalation — that is a normal unresolved issue, not an ignored prior ticket
- Ticket with fewer than 2 identifiable facts → populate missing_info with specific items needed
- Do NOT add missing_info items for details already present in the ticket (account_id, issue type, steps already tried)
""".strip()


RESOLUTION_SYSTEM = """
You are a support operations planner.
Based ONLY on the ticket, retrieved KB evidence, account context, and user memory:
- choose the correct route_to_team
- choose the recommended action
- write concise internal notes (2-3 sentences)
- decide whether clarification is needed

Rules:
- Legal or regulatory mentions → route_to_team MUST be human_specialist
- priority=critical → escalation is mandatory; prefer human_specialist if evidence is insufficient
- Use long-term user history to inform routing consistency
- Prefer clarification over false certainty
- Prefer escalation over unsupported commitments
- Tool and retrieval data overrides model intuition when they conflict
""".strip()


DRAFTING_SYSTEM = """
You are a customer support response writer.
Write a concise, professional reply grounded only in the evidence provided.

Rules:
- Apply the tone and style instruction exactly as given — do not deviate
- Do NOT promise refunds, credits, deadlines, or technical actions unless explicitly evidenced
- If information is missing, ask for it clearly and specifically
- If escalation is required, describe next steps conservatively — no overpromising
- Never use defensive language, shift blame, or minimise the customer concern
- Keep the reply under 120 words unless the situation requires more detail
""".strip()


REVISION_SYSTEM = """
You are revising a customer support response based on reviewer feedback.
Preserve factual correctness. Apply the reviewer's feedback precisely.
Do not introduce claims not supported by the original evidence.
Keep the revised reply concise and grounded.
""".strip()





print("Prompts defined: TRIAGE_SYSTEM, RESOLUTION_SYSTEM, DRAFTING_SYSTEM, REVISION_SYSTEM.")

print("Prompts defined: TRIAGE_SYSTEM, RESOLUTION_SYSTEM, DRAFTING_SYSTEM, REVISION_SYSTEM.")

## Cell 12 — Tool Layer (5 tools)

In [ ]:
@tool
def search_kb(query: str) -> List[dict]:
    """Search the internal support knowledge base and return the most relevant chunks."""
    docs = RETRIEVER.invoke(query)
    return [{"text": d.page_content, "metadata": d.metadata} for d in docs]


@tool
def lookup_account_context(account_id: str) -> Dict[str, Any]:
    """Look up account and open-ticket context by account ID (e.g. acct_1001)."""
    result = ACCOUNT_CONTEXT.get(account_id, {})
    if not result:
        # Try case-insensitive lookup
        result = ACCOUNT_CONTEXT.get(account_id.lower(), {})
    return result


@tool
def detect_escalation_risk(text: str) -> bool:
    """Detect whether a ticket contains strong escalation or regulatory risk signals."""
    text_l = text.lower()
    signals = [
        "cancel", "lawyer", "regulator", "ombudsman", "complaint", "authority",
        "unacceptable", "still not fixed", "nobody replied", "escalate this",
        "report this", "legal action", "consumer authority", "done with",
    ]
    return any(k in text_l for k in signals)


@tool
def priority_score(text: str) -> str:
    """Estimate support priority from deterministic urgency signals in the ticket text."""
    text_l = text.lower()
    if any(k in text_l for k in [
        "down", "outage", "cannot work", "all users affected",
        "regulator", "legal action", "consumer authority",
    ]):
        return "critical"
    if any(k in text_l for k in [
        "charged twice", "cancel", "lawyer", "urgent", "nobody replied", "done with",
    ]):
        return "high"
    if any(k in text_l for k in [
        "not working", "issue", "refund", "login", "log out", "logging me out", "cannot log",
    ]):
        return "medium"
    return "low"


@tool
def route_policy_lookup(category: str, priority: str) -> Dict[str, Any]:
    """Return the routing policy rule for a category + priority combination."""
    # Exact match
    for rule in ROUTING_RULES:
        if rule.get("trigger_category") == category and rule.get("priority") == priority:
            return rule
    # Category-only fallback (first matching category)
    for rule in ROUTING_RULES:
        if rule.get("trigger_category") == category:
            return rule
    # Unknown category fallback
    for rule in ROUTING_RULES:
        if rule.get("trigger_category") == "unknown":
            return rule
    return {"route_to_team": "human_specialist", "recommended_action": "manual_review"}


print("Tools defined:")
for t in [search_kb, lookup_account_context, detect_escalation_risk, priority_score, route_policy_lookup]:
    print(f"  - {t.name}")


## Cell 13 — Reliability: Retry Wrapper

In [ ]:
def with_retry(fn, retries: int = 2, delay: float = 1.0):
    """Call fn() with up to `retries` retries on exception."""
    for attempt in range(retries + 1):
        try:
            return fn()
        except Exception as e:
            if attempt == retries:
                raise
            print(f"  [RETRY] attempt {attempt + 1} failed ({e}), retrying in {delay}s...")
            time.sleep(delay)


## Cell 14 — Agent Nodes

In [ ]:
# ── Node 1: Load Long-Term Memory ────────────────────────────────────────────

def load_memory_node(state: TicketState) -> dict:
    """
    First node in the graph. Extracts account_id from user_request via regex,
    then pre-loads that user's long-term preferences, history, and global rules
    into state['memory_context'] so all downstream nodes can access them.
    """
    user_request = state.get("user_request", "")
    match = re.search(r"acct_\d+", user_request, re.IGNORECASE)
    account_id = match.group(0).lower() if match else "anonymous"

    try:
        user_prefs   = load_memory(MEMORY_CONN, f"user:{account_id}:preferences")
        user_history = load_memory(MEMORY_CONN, f"user:{account_id}:history")
        global_prefs = load_memory(MEMORY_CONN, "global:routing_preferences")
        global_tones = load_memory(MEMORY_CONN, "global:tone_rules")
    except Exception as e:
        print(f"[WARNING] Memory pre-load failed: {e}")
        user_prefs = user_history = global_prefs = global_tones = {}

    memory_context = {
        "account_id":        account_id,
        "user_preferences":  user_prefs,
        "user_history":      user_history,
        "global_routing":    global_prefs,
        "global_tones":      global_tones,
    }

    print(f"  [load_memory] account_id={account_id}  "
          f"prefs={list(user_prefs.keys())}  history={list(user_history.keys())}")

    return {
        "memory_context": memory_context,
        "status": "memory_loaded",
        "errors": state.get("errors", []),
    }


# ── Node 2: Triage Agent ──────────────────────────────────────────────────────

def triage_node(state: TicketState) -> dict:
    """
    Classifies the raw ticket into structured metadata.
    Uses triage_llm (structured output). Falls back to safe defaults on error.
    Ensures account_id from memory_context is in extracted_entities if LLM missed it.
    """
    errors = list(state.get("errors", []))

    def run():
        return triage_llm.invoke([
            SystemMessage(content=TRIAGE_SYSTEM),
            HumanMessage(content=state["user_request"]),
        ])

    try:
        result = with_retry(run)
    except Exception as e:
        errors.append(f"triage_node: {e}")
        return {
            "category": "unknown", "subcategory": "unknown",
            "priority": "medium",  "sentiment": "neutral",
            "requires_escalation": False, "missing_info": [],
            "extracted_entities": {},
            "messages": state.get("messages", []) + [
                {"role": "user", "content": state["user_request"]}
            ],
            "status": "triage_failed", "errors": errors,
        }

    # Guarantee account_id is present — use memory_context fallback if LLM missed it
    entities = dict(result.extracted_entities)
    if "account_id" not in entities:
        prefetched = state.get("memory_context", {}).get("account_id", "")
        if prefetched and prefetched != "anonymous":
            entities["account_id"] = prefetched

    print(f"  [triage] category={result.category}  priority={result.priority}"
          f"  sentiment={result.sentiment}  escalation={result.requires_escalation}"
          f"  account_id={entities.get('account_id','—')}")

    return {
        "messages": state.get("messages", []) + [
            {"role": "user", "content": state["user_request"]}
        ],
        "category":            result.category,
        "subcategory":         result.subcategory,
        "priority":            result.priority,
        "sentiment":           result.sentiment,
        "requires_escalation": result.requires_escalation,
        "missing_info":        result.missing_info,
        "extracted_entities":  entities,
        "status": "triaged",
        "errors": errors,
    }


# ── Conditional edge after triage ─────────────────────────────────────────────

def after_triage_route(state: TicketState) -> Literal["clarify", "retrieve"]:
    missing = state.get("missing_info", [])
    if missing and len(missing) >= 2:
        print(f"  [route] → clarify ({len(missing)} missing fields)")
        return "clarify"
    print("  [route] → retrieve")
    return "retrieve"


# ── Node 3: Clarification ─────────────────────────────────────────────────────

def clarify_node(state: TicketState) -> dict:
    missing = ", ".join(state.get("missing_info", [])) or "a few more details"
    reply = (
        f"Thank you for reaching out. To help you as quickly as possible, "
        f"could you please provide: {missing}? "
        f"Once we have these details we will route your case to the right team."
    )
    return {
        "messages":    state.get("messages", []) + [{"role": "assistant", "content": reply}],
        "reply_draft": reply,
        "final_reply": reply,
        "status":      "needs_clarification",
    }


# ── Node 4: Retrieval Agent ───────────────────────────────────────────────────

def retrieval_node(state: TicketState) -> dict:
    """
    Uses 4 tools: search_kb, lookup_account_context, detect_escalation_risk, priority_score.
    """
    account_id = (
        state.get("extracted_entities", {}).get("account_id")
        or state.get("memory_context", {}).get("account_id", "")
    )
    kb_query = f"{state.get('category', '')} {state['user_request']}"

    kb_results         = search_kb.invoke({"query": kb_query})
    account_ctx        = lookup_account_context.invoke({"account_id": account_id}) if account_id else {}
    escalation_signal  = detect_escalation_risk.invoke({"text": state["user_request"]})
    priority_signal    = priority_score.invoke({"text": state["user_request"]})

    kb_summary = " | ".join([r["text"][:180] for r in kb_results[:3]]) if kb_results else ""

    final_escalation = bool(state.get("requires_escalation", False) or escalation_signal)

    # Use the lower of LLM triage priority and the deterministic tool signal.
    # The tool is rule-based and immune to LLM over-escalation; it prevents priority
    # inflation caused by phrases like "still cannot X" being misread as high-urgency.
    _rank = {"low": 0, "medium": 1, "high": 2, "critical": 3}
    llm_priority  = state.get("priority") or priority_signal
    final_priority = (
        llm_priority
        if _rank.get(llm_priority, 1) <= _rank.get(priority_signal, 1)
        else priority_signal
    )

    print(f"  [retrieve] kb_hits={len(kb_results)}  account_found={bool(account_ctx)}"
          f"  escalation={final_escalation}  priority_signal={priority_signal}")

    return {
        "kb_results":          kb_results,
        "kb_summary":          kb_summary,
        "account_context":     account_ctx,
        "requires_escalation": final_escalation,
        "priority":            final_priority,
        "status": "retrieved",
    }


# ── Node 5: Resolution Agent ──────────────────────────────────────────────────

def resolution_node(state: TicketState) -> dict:
    """
    Reads fresh long-term memory, calls route_policy_lookup tool,
    then uses plan_llm to produce a structured ResolutionPlan.
    """
    user_id = (
        state.get("extracted_entities", {}).get("account_id")
        or state.get("memory_context", {}).get("account_id", "anonymous")
    )

    # Read-before-act: fresh memory read + fallback to pre-loaded context
    user_prefs   = load_memory(MEMORY_CONN, f"user:{user_id}:preferences") \
                   or state.get("memory_context", {}).get("user_preferences", {})
    user_history = load_memory(MEMORY_CONN, f"user:{user_id}:history") \
                   or state.get("memory_context", {}).get("user_history", {})

    policy_rule = route_policy_lookup.invoke({
        "category": state.get("category", "unknown"),
        "priority": state.get("priority", "medium"),
    })

    prompt = (
        f"User request      : {state['user_request']}\n"
        f"Category          : {state.get('category')}\n"
        f"Subcategory       : {state.get('subcategory')}\n"
        f"Priority          : {state.get('priority')}\n"
        f"Sentiment         : {state.get('sentiment')}\n"
        f"Requires escalation: {state.get('requires_escalation')}\n"
        f"Missing info      : {state.get('missing_info')}\n"
        f"KB summary        : {state.get('kb_summary')}\n"
        f"Full KB results   :\n{json.dumps(state.get('kb_results', [])[:3], indent=2)}\n"
        f"Account context   :\n{json.dumps(state.get('account_context', {}), indent=2)}\n"
        f"Policy rule       :\n{json.dumps(policy_rule, indent=2)}\n"
        f"User preferences  : {json.dumps(user_prefs)}\n"
        f"User history      : {json.dumps(user_history)}"
    )

    errors = list(state.get("errors", []))

    def run():
        return plan_llm.invoke([
            SystemMessage(content=RESOLUTION_SYSTEM),
            HumanMessage(content=prompt),
        ])

    try:
        result = with_retry(run)
        route_to_team      = result.route_to_team
        recommended_action = result.recommended_action
        internal_notes     = result.internal_notes
        clarification      = result.should_ask_for_clarification
    except Exception as e:
        errors.append(f"resolution_node: {e}")
        route_to_team      = policy_rule.get("route_to_team", "human_specialist")
        recommended_action = policy_rule.get("recommended_action", "manual_review")
        internal_notes     = "Automatic planning failed — routed for manual review."
        clarification      = False

    # Escalation override: force human_specialist if critical or escalation required and no route set
    if (state.get("requires_escalation") or state.get("priority") == "critical") and not route_to_team:
        route_to_team = "human_specialist"

    print(f"  [resolve] route={route_to_team}  action={recommended_action[:50]}")

    return {
        "route_to_team":              route_to_team,
        "recommended_action":         recommended_action,
        "internal_notes":             internal_notes,
        "should_ask_for_clarification": clarification,
        "status": "planned",
        "errors": errors,
    }


# ── Node 6: Drafting Agent ────────────────────────────────────────────────────

def draft_node(state: TicketState) -> dict:
    """
    Writes the customer-facing reply.
    Reads tone and reply_style from pre-loaded memory_context;
    falls back to a fresh DB read if memory_context is empty.
    """
    memory_ctx = state.get("memory_context", {})
    user_prefs = memory_ctx.get("user_preferences", {})

    if not user_prefs:
        user_id = (
            state.get("extracted_entities", {}).get("account_id")
            or memory_ctx.get("account_id", "anonymous")
        )
        user_prefs = load_memory(MEMORY_CONN, f"user:{user_id}:preferences")

    tone        = user_prefs.get("tone", DEFAULT_TONE)
    reply_style = user_prefs.get("reply_style", "")
    tone_instruction = f"Tone: {tone}."
    if reply_style:
        tone_instruction += f" Reply style: {reply_style}."

    prompt = (
        f"{tone_instruction}\n\n"
        f"User request      : {state['user_request']}\n"
        f"Category          : {state.get('category')}\n"
        f"Priority          : {state.get('priority')}\n"
        f"Sentiment         : {state.get('sentiment')}\n"
        f"Requires escalation: {state.get('requires_escalation')}\n"
        f"Missing info      : {state.get('missing_info')}\n"
        f"Route to team     : {state.get('route_to_team')}\n"
        f"Recommended action: {state.get('recommended_action')}\n"
        f"Internal notes    : {state.get('internal_notes')}\n"
        f"KB evidence       :\n{json.dumps(state.get('kb_results', [])[:3], indent=2)}\n"
        f"Account context   :\n{json.dumps(state.get('account_context', {}), indent=2)}\n\n"
        f"Rules:\n"
        f"- Ground the reply only in the provided evidence.\n"
        f"- Apply tone and style exactly.\n"
        f"- Do not promise unsupported actions.\n"
        f"- If escalation is required, explain next steps conservatively."
    )

    errors = list(state.get("errors", []))

    def run():
        return base_llm.invoke([
            SystemMessage(content=DRAFTING_SYSTEM),
            HumanMessage(content=prompt),
        ])

    try:
        response = with_retry(run)
        reply = response.content
    except Exception as e:
        errors.append(f"draft_node: {e}")
        reply = (
            "Thank you for contacting support. We are reviewing your case and will route it "
            "to the appropriate team. Please reply with any additional details if needed."
        )

    print(f"  [draft] tone={tone}  length={len(reply)} chars")

    return {
        "messages":    state.get("messages", []) + [{"role": "assistant", "content": reply}],
        "reply_draft": reply,
        "status": "drafted",
        "errors": errors,
    }


# ── Node 7: Review Node (HITL) ────────────────────────────────────────────────

def review_node(state: TicketState) -> Command[Literal["finalize", "revise", "manual_escalation"]]:
    """
    Pauses execution for human review. The interrupt payload shows the draft and metadata.
    Resumes based on human decision:
      - "approve"             → finalize
      - "revise: <feedback>"  → revise (feedback injected into state)
      - "escalate_manually"   → manual_escalation
    """
    review_payload = {
        "question":           "Review the proposed support response and choose an action.",
        "category":           state.get("category"),
        "priority":           state.get("priority"),
        "route_to_team":      state.get("route_to_team"),
        "requires_escalation": state.get("requires_escalation"),
        "reply_draft":        state.get("reply_draft"),
        "instructions":       "Reply with one of:  approve  |  revise: <feedback>  |  escalate_manually",
    }

    print("  [review] HITL interrupt — waiting for human decision...")
    decision = interrupt(review_payload)

    if isinstance(decision, str):
        d = decision.strip().lower()
        if d == "approve":
            print("  [review] Decision: APPROVE")
            return Command(goto="finalize")
        if d.startswith("revise:"):
            print(f"  [review] Decision: REVISE — {decision[7:50]}...")
            return Command(update={"human_feedback": decision}, goto="revise")
        if d == "escalate_manually":
            print("  [review] Decision: ESCALATE MANUALLY")
            return Command(update={"human_feedback": decision}, goto="manual_escalation")

    print("  [review] Unrecognised decision — defaulting to approve.")
    return Command(update={"human_feedback": "auto-approved"}, goto="finalize")


# ── Node 8: Revise ────────────────────────────────────────────────────────────

def revise_node(state: TicketState) -> dict:
    """Revises the draft based on reviewer feedback."""
    prompt = (
        f"Current draft:\n{state.get('reply_draft')}\n\n"
        f"Reviewer feedback:\n{state.get('human_feedback')}"
    )
    errors = list(state.get("errors", []))

    def run():
        return base_llm.invoke([
            SystemMessage(content=REVISION_SYSTEM),
            HumanMessage(content=prompt),
        ])

    try:
        response = with_retry(run)
        revised = response.content
    except Exception as e:
        errors.append(f"revise_node: {e}")
        revised = state.get("reply_draft", "")

    print(f"  [revise] Revised draft: {revised[:80]}...")

    return {
        "messages":    state.get("messages", []) + [{"role": "assistant", "content": revised}],
        "reply_draft": revised,
        "status": "revised",
        "errors": errors,
    }


# ── Node 9: Finalize ──────────────────────────────────────────────────────────

def finalize_node(state: TicketState) -> dict:
    """
    Approves the draft and writes updated facts back to long-term memory.
    Memory write failures are non-fatal (logged, not raised).
    """
    user_id = (
        state.get("extracted_entities", {}).get("account_id")
        or state.get("memory_context", {}).get("account_id", "anonymous")
    )

    try:
        if state.get("sentiment") in ("angry", "frustrated"):
            save_memory(MEMORY_CONN, f"user:{user_id}:preferences", "tone", "empathetic")
        if state.get("requires_escalation"):
            save_memory(MEMORY_CONN, f"user:{user_id}:history", "escalation_flag", True)
        if state.get("category"):
            save_memory(MEMORY_CONN, f"user:{user_id}:history", "last_category", state["category"])
        if state.get("route_to_team"):
            save_memory(MEMORY_CONN, f"user:{user_id}:history", "last_known_route", state["route_to_team"])
        print(f"  [finalize] Memory written back for {user_id}.")
    except Exception as e:
        print(f"  [WARNING] Memory write-back failed: {e}")

    return {
        "final_reply":    state.get("reply_draft"),
        "human_approved": True,
        "status":         "approved",
    }


# ── Node 10: Manual Escalation ────────────────────────────────────────────────


def manual_escalation_node(state: TicketState) -> dict:
    reply = (
        "Your case has been escalated for manual handling by a support specialist. "
        "We will review the details and follow up through your preferred contact channel."
    )
    print("  [manual_escalation] Case escalated to human_specialist.")

    return {
        "final_reply":    reply,
        "human_approved": False,
        "status":         "manual_escalation",
        "route_to_team":  "human_specialist",
    }


print("All 10 node functions defined.")


## Cell 15 — Graph Construction

In [ ]:
# LangGraph checkpointer (SQLite for thread-state persistence)
# NOTE: SqliteSaver.from_conn_string() returns a context manager in newer versions
# and cannot be assigned directly. Use the constructor with a raw sqlite3 connection instead.
try:
    _cp_conn = sqlite3.connect("workflow_state.sqlite", check_same_thread=False)
    checkpointer = SqliteSaver(_cp_conn)
    print("Checkpointer: SqliteSaver (workflow_state.sqlite)")
except Exception:
    from langgraph.checkpoint.memory import MemorySaver
    checkpointer = MemorySaver()
    print("[WARNING] Using MemorySaver — graph state will not persist across restarts.")

# Build the stateful graph
builder = StateGraph(TicketState)

builder.add_node("load_memory",       load_memory_node)
builder.add_node("triage",            triage_node)
builder.add_node("clarify",           clarify_node)
builder.add_node("retrieve",          retrieval_node)
builder.add_node("resolve",           resolution_node)
builder.add_node("draft",             draft_node)
builder.add_node("review",            review_node)
builder.add_node("revise",            revise_node)
builder.add_node("finalize",          finalize_node)
builder.add_node("manual_escalation", manual_escalation_node)

builder.add_edge(START,           "load_memory")
builder.add_edge("load_memory",   "triage")
builder.add_conditional_edges("triage", after_triage_route, ["clarify", "retrieve"])
builder.add_edge("clarify",       END)
builder.add_edge("retrieve",      "resolve")
builder.add_edge("resolve",       "draft")
builder.add_edge("draft",         "review")
builder.add_edge("revise",        "review")
builder.add_edge("finalize",      END)
builder.add_edge("manual_escalation", END)

graph = builder.compile(checkpointer=checkpointer)

print("Graph compiled successfully.")
print("Nodes:", [n for n in builder.nodes if n not in ("__start__", "__end__")])


## Cell 16 — Core Functions: `execute_workflow`, `resume_workflow`, Helpers

In [ ]:
def execute_workflow(user_request: str) -> dict:
    """
    Required core function (assignment spec).
    Initialises the graph with a fresh thread and begins execution.

    Returns:
        {thread_id, result}
        result contains '__interrupt__' (tuple of Interrupt objects) when HITL paused execution,
        or the final TicketState dict when the graph reached END without interrupting.
    """
    thread_id = str(uuid.uuid4())
    config    = {"configurable": {"thread_id": thread_id}}

    result = graph.invoke(
        {
            "thread_id":      thread_id,
            "user_request":   user_request,
            "messages":       [{"role": "user", "content": user_request}],
            "human_approved": False,
            "status":         "started",
            "errors":         [],
        },
        config=config,
    )
    return {"thread_id": thread_id, "result": result}


def resume_workflow(thread_id: str, decision: str) -> dict:
    """
    Resume a graph paused at HITL interrupt with a human decision string.
    Decision options:
      - "approve"
      - "revise: <your feedback here>"
      - "escalate_manually"
    """
    config = {"configurable": {"thread_id": thread_id}}
    return graph.invoke(Command(resume=decision), config=config)


def get_interrupt_payload(run_result: dict) -> dict:
    """
    Safely unpack the interrupt payload from a graph result dict.
    graph.invoke() returns {"__interrupt__": (Interrupt(value={...}),)} when paused.
    """
    raw = run_result.get("__interrupt__")
    if raw is None:
        return {}
    if isinstance(raw, (list, tuple)) and raw:
        item = raw[0]
        return item.value if hasattr(item, "value") else dict(item)
    return {}


def print_summary(state: dict, label: str = ""):
    """Print a compact human-readable summary of the final state."""
    if label:
        print(f"\n{'='*62}\n  {label}\n{'='*62}")
    fields = [
        "status", "category", "subcategory", "priority", "sentiment",
        "requires_escalation", "route_to_team", "recommended_action",
        "human_approved", "errors",
    ]
    for f in fields:
        if f in state:
            print(f"  {f:<26}: {state[f]}")
    reply = state.get("final_reply") or state.get("reply_draft", "")
    if reply:
        print(f"\n  FINAL REPLY:\n  {'-'*50}")
        for line in reply.strip().split("\n"):
            print(f"  {line}")
    print()


def print_interrupt(run_result: dict):
    """Display the HITL interrupt payload in a readable format."""
    payload = get_interrupt_payload(run_result)
    if not payload:
        print("  No interrupt payload found.")
        return
    print("\n  --- HUMAN REVIEW REQUIRED ---")
    for k, v in payload.items():
        if k == "reply_draft":
            print(f"  {k}:")
            for line in str(v).strip().split("\n"):
                print(f"    {line}")
        else:
            print(f"  {k:<22}: {v}")
    print()


def print_user_memory(user_id: str):
    """Display long-term memory for a given user account."""
    print(f"\n  Long-term memory — {user_id}:")
    for ns in ["preferences", "history"]:
        mem = load_memory(MEMORY_CONN, f"user:{user_id}:{ns}")
        print(f"    [{ns}] {mem if mem else '(empty)'}")


print("Core functions ready: execute_workflow, resume_workflow, get_interrupt_payload")
print("Helpers: print_summary, print_interrupt, print_user_memory")


## Cell 17 — Graph Visualisation

In [ ]:
# Text Mermaid representation (always works)
try:
    print("Graph structure (Mermaid):\n")
    print(graph.get_graph().draw_mermaid())
except Exception as e:
    print(f"Mermaid text not available: {e}")

# PNG render (requires graphviz — uncomment if available in your Colab session)
# from IPython.display import Image, display
# try:
#     display(Image(graph.get_graph().draw_mermaid_png()))
# except Exception as e:
#     print(f"PNG render failed: {e}")


## Cell 18 — `run_test_case` Helper

Wraps the full HITL flow for a single test case:
`execute_workflow → inspect interrupt → resume_workflow → print summary → inspect memory`.


In [ ]:
def run_test_case(
    test_id: str,
    human_decision: str = None,
    verbose: bool = True,
) -> dict:
    """
    Execute a single test case by ID.
    Handles the full HITL flow automatically.

    Args:
        test_id       : ID from test_cases in project_data.json
        human_decision: Override simulate_human_decision from test case data. If None, uses data value.
        verbose       : Print intermediate state.

    Returns:
        {thread_id, initial_result, final_result, test_meta}
    """
    test = next((t for t in TEST_CASES if t["id"] == test_id), None)
    if test is None:
        raise ValueError(f"Test case '{test_id}' not found.")

    decision = human_decision or test.get("simulate_human_decision", "approve")

    sep = "=" * 65
    print(f"\n{sep}")
    print(f"  TEST : {test['id']}")
    print(f"{sep}")
    print(f"  INPUT    : {test['input']}")
    print(f"  EXPECTED : category={test['expected_category']}  "
          f"priority={test['expected_priority']}")
    print(f"  EXPECTED : route={test['expected_route']}  "
          f"escalation={test['expected_requires_escalation']}")
    print(f"  DECISION : {decision[:70]}")
    print(f"  NOTES    : {test.get('notes', '')}")
    print()

    # Step 1 — Execute (runs until interrupt or END)
    print(">> STEP 1: execute_workflow()")
    run          = execute_workflow(test["input"])
    thread_id    = run["thread_id"]
    init_result  = run["result"]
    print(f"   thread_id = {thread_id}")

    # Step 2 — Check for HITL interrupt
    payload = get_interrupt_payload(init_result)

    if payload:
        if verbose:
            print_interrupt(init_result)
        else:
            draft_preview = str(payload.get("reply_draft", ""))[:80]
            print(f"   Interrupted at review_node. Draft: {draft_preview}...")

        # Step 3 — Resume with human decision
        print(f">> STEP 2: resume_workflow() — decision: {decision[:70]}")
        final_result = resume_workflow(thread_id, decision)
    else:
        # No interrupt — clarify path or other direct END
        print("   No interrupt — workflow completed without HITL (clarify path or direct END).")
        final_result = init_result

    # Step 4 — Summary
    if verbose:
        print_summary(final_result, label=f"RESULT: {test['id']}")

    # Step 5 — Memory inspection
    account_id = (
        final_result.get("extracted_entities", {}).get("account_id")
        or final_result.get("memory_context", {}).get("account_id", "")
    )
    if account_id and account_id not in ("anonymous", ""):
        print_user_memory(account_id)

    return {
        "thread_id":     thread_id,
        "initial_result": init_result,
        "final_result":  final_result,
        "test_meta":     test,
    }


print("run_test_case ready.")


## Test 1 — Duplicate Billing Charge

**Expected**: category=billing · priority=high · route=billing_support · HITL: **approve**  
Account acct_1001 has `refund_pending=True` and existing KB guidance on duplicate charges.

In [ ]:
result_ng = run_test_case("test_01_duplicate_billing")


## Test 2 — Business Office Outage

**Expected**: category=technical · priority=critical · route=technical_support · HITL: **escalate_manually**  
Business-plus account (acct_1002) with `known_outage=True` in account context.

In [ ]:
result_ge = run_test_case("test_02_business_outage")


## Test 3 — Login / MFA Failure

**Expected**: category=account_access · priority=medium · route=account_access_support · HITL: **approve**  
acct_1008 has prior login failure in history. KB articles: login-reset + MFA-lockout.

In [ ]:
result_ss = run_test_case("test_03_login_access")


## Test 4 — Cancellation Threat (Revision Loop)

**Expected**: category=cancellation · priority=high · route=retention_team · HITL: **revise**  
acct_1004 has `recent_escalation_flag=True` and `tone=empathetic` in long-term memory.  
After finalize, memory write-back should record `tone=empathetic`.

In [ ]:
result_at = run_test_case("test_04_cancellation_threat")


## Test 5 — Incomplete / Vague Ticket

**Expected**: triage detects ≥ 2 missing fields → routes to **clarify node** (no HITL).  
Demonstrates the conditional edge `after_triage_route → clarify`.

In [ ]:
result_et = run_test_case("test_05_incomplete_ticket")


## Test 6 — App Logout Revision Loop

**Expected**: category=technical · priority=medium · route=technical_support · HITL: **revise**  
Uses the new `kb_technical_app_session_logout` article.  
Message history must contain both the original draft and the revised version.

In [ ]:
result_on = run_test_case("test_06_app_logout_revision")

# Show message history to verify both draft and revision are present
print("\nMessage history (short-term memory):")
msgs = result_on.get("final_result", {}).get("messages", [])
for i, m in enumerate(msgs):
    role    = m.get("role", "?")
    preview = str(m.get("content", ""))[:120]
    print(f"  [{i}] {role}: {preview}...")


## Test 7 — Legal / Regulatory Threat

**Expected**: category=complaint · priority=critical · route=human_specialist · HITL: **escalate_manually**  
acct_1009 has `tone=formal_neutral` in long-term memory.  
Triggers `kb_legal_regulatory_escalation` KB article and specialist routing.

In [ ]:
result_le = run_test_case("test_07_legal_regulatory_threat")


## Memory Inspection — All Accounts After Full Test Run

In [ ]:
print("=" * 65)
print("  LONG-TERM MEMORY STATE — all accounts")
print("=" * 65)

for acct_id in sorted(ACCOUNT_CONTEXT.keys()):
    prefs   = load_memory(MEMORY_CONN, f"user:{acct_id}:preferences")
    history = load_memory(MEMORY_CONN, f"user:{acct_id}:history")
    if prefs or history:
        print(f"\n  {acct_id}:")
        if prefs:
            print(f"    [preferences] {prefs}")
        if history:
            print(f"    [history]     {history}")

print("\n  Global routing preferences:")
print(" ", load_memory(MEMORY_CONN, "global:routing_preferences"))
print("\n  Global tone rules:")
print(" ", load_memory(MEMORY_CONN, "global:tone_rules"))


## Grader Notes

### Agents (5 + HITL)
| Node | Agent Role |
|---|---|
| `load_memory_node` | Pre-loads per-user long-term memory before any agent runs |
| `triage_node` | Classifies ticket into structured metadata — `triage_llm` (structured output) |
| `retrieval_node` | Gathers KB evidence + account context + urgency signals via tools |
| `resolution_node` | Plans route + action using KB + memory + account context — `plan_llm` (structured output) |
| `draft_node` | Writes tone-aware, grounded customer reply — reads memory for tone |
| `review_node` | **HITL** interrupt — pauses for approve / revise / escalate_manually |

### Tools (5)
| Tool | Type |
|---|---|
| `search_kb` | FAISS retriever — RAG over 17 KB articles |
| `lookup_account_context` | Custom — account/ticket context dict lookup |
| `detect_escalation_risk` | Custom — deterministic keyword signal detection |
| `priority_score` | Custom — keyword-based priority estimation |
| `route_policy_lookup` | Custom — routing rule table lookup |

### Memory (3 layers)
- **Short-term**: `messages` field in `TicketState` — tiktoken-trimmed before every LLM call
- **Long-term**: SQLite `memory_store` — namespaced per user, seeded from `project_data.json`
  - Read-before-act in `resolution_node` and `draft_node`
  - Write-back in `finalize_node` (sentiment/escalation/category/route)
- **RAG**: Runtime FAISS vector store built from 17 KB articles at startup

### HITL
`review_node` uses `interrupt()` to pause. `resume_workflow(thread_id, decision)` resumes.
Three paths: `approve` → finalize | `revise: <feedback>` → revise → review | `escalate_manually` → manual_escalation

### Test Cases (7)
| # | Scenario | HITL Path |
|---|---|---|
| 1 | Duplicate billing — acct_1001 | approve |
| 2 | Business outage — acct_1002 | escalate_manually |
| 3 | Login / MFA failure — acct_1008 | approve |
| 4 | Cancellation threat — acct_1004 | revise |
| 5 | Vague / incomplete ticket | clarify (no HITL) |
| 6 | App logout — acct_1007 | revise (revision loop) |
| 7 | Legal / regulatory threat — acct_1009 | escalate_manually |


## Automated Test Scoring

Compares each test's actual output against the expected values in `project_data.json`.
Checks: **category**, **priority**, **route_to_team**, **requires_escalation**, and whether a **final reply** was produced.


In [ ]:
# ── Automated Test Scorer ────────────────────────────────────────────────────
# Maps result variables to their test IDs (order matches test execution order).
ALL_RESULTS = {
    "test_01_duplicate_billing":       result_ng,
    "test_02_business_outage":         result_ge,
    "test_03_login_access":            result_ss,
    "test_04_cancellation_threat":     result_at,
    "test_05_incomplete_ticket":       result_et,
    "test_06_app_logout_revision":     result_on,
    "test_07_legal_regulatory_threat": result_le,
}

# Fields to compare: (result_key, expected_key, display_label)
CHECKS = [
    ("category",           "expected_category",           "category"),
    ("priority",           "expected_priority",           "priority"),
    ("route_to_team",      "expected_route",              "route_to_team"),
    ("requires_escalation","expected_requires_escalation","requires_escalation"),
]

PASS = "PASS"
FAIL = "FAIL"
NA   = "N/A "

col_w = 26   # width for field columns

def score_result(result: dict) -> list[dict]:
    """Return a list of check dicts for one test result."""
    meta   = result["test_meta"]
    state  = result["final_result"]
    checks = []
    for state_key, meta_key, label in CHECKS:
        actual   = state.get(state_key)
        expected = meta.get(meta_key)
        if actual is None:
            status = NA
        elif actual == expected:
            status = PASS
        else:
            status = FAIL
        checks.append({
            "label":    label,
            "expected": expected,
            "actual":   actual,
            "status":   status,
        })
    # Bonus check: a non-empty final reply was produced
    has_reply = bool(state.get("final_reply") or state.get("reply_draft"))
    checks.append({
        "label":    "has_final_reply",
        "expected": True,
        "actual":   has_reply,
        "status":   PASS if has_reply else FAIL,
    })
    return checks


# ── Run scorer and print table ───────────────────────────────────────────────
header = f"{'TEST':<42} {'FIELD':<22} {'EXPECTED':<22} {'ACTUAL':<22} STATUS"
divider = "-" * len(header)

print("=" * len(header))
print("  AUTOMATED TEST SCORES")
print("=" * len(header))
print(header)
print(divider)

total = passed = 0
summary_rows = []

for test_id, result in ALL_RESULTS.items():
    checks = score_result(result)
    test_passed = sum(1 for c in checks if c["status"] == PASS)
    test_total  = len(checks)
    total  += test_total
    passed += test_passed

    for i, c in enumerate(checks):
        label_col = test_id if i == 0 else ""
        status_sym = {"PASS": "✓", "FAIL": "✗", "N/A ": "–"}.get(c["status"], "?")
        print(
            f"  {label_col:<40} {c['label']:<22} "
            f"{str(c['expected']):<22} {str(c['actual']):<22} "
            f"{status_sym} {c['status']}"
        )
    summary_rows.append((test_id, test_passed, test_total))
    print(divider)

pct = round(100 * passed / total) if total else 0
print(f"\n  OVERALL: {passed}/{total} checks passed  ({pct}%)")
print()
print(f"  Per-test breakdown:")
for tid, tp, tt in summary_rows:
    bar = "█" * tp + "░" * (tt - tp)
    print(f"    {tid:<42} {tp}/{tt}  [{bar}]")
print()


## 7. Interactive System Demo

**Try it yourself!** Run the python cell below to chat directly with the multi-agent triage system. 

The system will pause and expose its internal mechanics (**🧠 System Analysis**, **🏢 Routing & Notes**, and **✍️ Draft**) before explicitly asking you, the Manager, for a **🕵️ Reviewer Intervention** decision.



In [ ]:
import IPython.display as display

def display_ticket_state(state):
    """Helper to beautifully format the internal state of the agent system"""
    entities = state.get('extracted_entities', {})
    
    display.display(display.Markdown("---"))
    display.display(display.Markdown("### 🧠 System Analysis"))
    display.display(display.Markdown(f"- **Category:** `{state.get('category')} / {state.get('subcategory')}`"))
    display.display(display.Markdown(f"- **Priority:** `{state.get('priority')}`"))
    display.display(display.Markdown(f"- **Extracted Entities:** `{entities}`"))
    display.display(display.Markdown(f"- **History Flag:** `{'Yes' if state.get('account_context') else 'No'}`"))
    
    display.display(display.Markdown("### 🏢 Internal Routing & Notes"))
    display.display(display.Markdown(f"- **Assigned Team:** `{state.get('route_to_team', 'Unassigned')}`"))
    display.display(display.Markdown(f"- **Recommended Action:** `{state.get('recommended_action', 'None')}`"))
    display.display(display.Markdown(f"- **Internal Notes:** _{state.get('internal_notes', 'None')}_"))
    
    display.display(display.Markdown("### ✍️ Generated Draft"))
    display.display(display.Markdown(f"> {state.get('reply_draft', '')}"))
    display.display(display.Markdown("---"))


def _is_graph_paused(thread_id: str) -> bool:
    """Check via graph.get_state() whether the thread is still waiting at an interrupt."""
    config = {"configurable": {"thread_id": thread_id}}
    snapshot = graph.get_state(config)
    return bool(snapshot.tasks)


def _ask_reviewer() -> str:
    """Prompt the reviewer and return a normalised decision string."""
    decision = input("Manager Decision (approve / revise / escalate_manually): ").strip().lower()
    if decision not in ['approve', 'revise', 'escalate_manually']:
        print("Invalid decision. Defaulting to 'approve'.")
        decision = "approve"
    if decision == "revise":
        feedback = input("Feedback for Agent revision: ").strip()
        decision = f"revise: {feedback}"
    return decision


def interactive_session():
    print("==================================================")
    print("🤖 Welcome to the Cortex Support Triage Terminal")
    print("Type 'quit' or 'exit' to stop.")
    print("==================================================\n")
    
    while True:
        user_msg = input("\n👤 Customer Support Request: ")
        if user_msg.lower() in ['quit', 'exit']:
            print("Ending session.")
            break
        if not user_msg.strip():
            continue
            
        print("\n⏳ System is processing (Agents are thinking)...")
        try:
            result = execute_workflow(user_msg)
        except Exception as e:
            print(f"❌ Error executing workflow: {e}")
            continue
        
        thread_id = result["thread_id"]
        state     = result.get('result', {})

        display_ticket_state(state)

        if _is_graph_paused(thread_id):
            # Use graph.get_state().tasks as the authoritative pause signal.
            # graph.invoke() return dict is unreliable for detecting mid-stream
            # interrupts in langgraph 0.2.x — __interrupt__ key is absent even
            # when the graph is still paused at review_node.
            final_state = state
            while _is_graph_paused(thread_id):
                display.display(display.Markdown("### 🕵️ Reviewer Intervention Required"))
                decision = _ask_reviewer()
                print(f"\n⏳ Resuming workflow with decision: '{decision}'...")
                try:
                    final_state = resume_workflow(thread_id, decision)
                except Exception as e:
                    print(f"❌ Error resuming workflow: {e}")
                    break
                if _is_graph_paused(thread_id):
                    display_ticket_state(final_state)

            display.display(display.Markdown("### ✅ Final Customer Response"))
            display.display(display.Markdown(
                f"**Sent to Customer:**\n\n{final_state.get('final_reply', 'No reply generated.')}"
            ))
            account_id = final_state.get('extracted_entities', {}).get('account_id')
            if account_id and account_id != 'anonymous':
                prefs = load_memory(MEMORY_CONN, f"user:{account_id}:preferences")
                hist  = load_memory(MEMORY_CONN, f"user:{account_id}:history")
                display.display(display.Markdown("### 💾 Written to Database (Long-Term Memory)"))
                display.display(display.Markdown(f"- **Preferences:** `{prefs}`\n- **History:** `{hist}`"))
        else:
            display.display(display.Markdown("### ✅ Final Customer Response (No Review Needed)"))
            display.display(display.Markdown(
                f"**Sent to Customer:**\n\n{state.get('final_reply', 'No reply generated.')}"
            ))
            account_id = state.get('extracted_entities', {}).get('account_id')
            if account_id and account_id != 'anonymous':
                prefs = load_memory(MEMORY_CONN, f"user:{account_id}:preferences")
                hist  = load_memory(MEMORY_CONN, f"user:{account_id}:history")
                display.display(display.Markdown("### 💾 Written to Database (Long-Term Memory)"))
                display.display(display.Markdown(f"- **Preferences:** `{prefs}`\n- **History:** `{hist}`"))

# Run the interactive session!
# Uncomment the line below to start chatting
# interactive_session()

**Suggested Scenarios:**
1. **Double Charge (Account: acct_1001)**: *"Hi, my account acct_1001 was charged twice for the Pro plan this month. Fix this!"*
2. **Critical Outage**: *"The entire production database is down and our app is returning 500 errors to everyone. Help immediately!"*
3. **Vague Request**: *"It's not working anymore."*
4. **Angry Escalate**: *"I have been waiting for weeks! Cancel my subscription and escalate this to legal immediately!"*

In [ ]:
interactive_session()